In [3]:
%matplotlib inline
import random
import torch


In [4]:
def synthtic_data(w,b,num_examples):
    x=torch.normal(0,1,(num_examples,len(w)))
    y=torch.matmul(x,w)+b
    y+=torch.normal(0,0.01,y.shape)
    return x,y.reshape((-1,1))

true_w=torch.tensor([2,-3.4])
true_b=4.2
features,lables=synthtic_data(true_w,true_b,1000)

In [8]:
#定义一个函数接取批量大小
def data_iter(batch_size,features,lables):
    num_examples=len(features)
    indices=list(range(num_examples))
    #随机顺序读取
    random.shuffle(indices)
    for i in range(0,num_examples,batch_size):
        batch_indices=torch.tensor(indices[i:min(i+batch_size,num_examples)])
        yield features[batch_indices],lables[batch_indices]
batch_size=10
for x,y in data_iter(batch_size,features,lables):
    print(x,'\n',y)
    break
        

tensor([[ 0.7773,  0.6181],
        [-0.6581, -0.6566],
        [ 0.7023, -2.8396],
        [ 1.1147, -0.9001],
        [ 1.7338,  0.2712],
        [ 1.9339, -1.5936],
        [ 1.1152, -0.4312],
        [ 0.5330, -1.1100],
        [-0.2250, -1.6505],
        [ 0.7498,  0.1208]]) 
 tensor([[ 3.6545],
        [ 5.1452],
        [15.2600],
        [ 9.4778],
        [ 6.7312],
        [13.4836],
        [ 7.8954],
        [ 9.0352],
        [ 9.3743],
        [ 5.2971]])


In [9]:
#定义初始化模型参数
w=torch.normal(0,0.01,size=(2,1),requires_grad=True)
b=torch.zeros(1,requires_grad=True)



In [10]:
#定义模型
def linreg(x,w,b):
    return torch.matmul(x,w)+b


In [11]:
def squared_loss(y_hat, y):
    """均方损失函数"""
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

In [12]:
def sgd(params, lr, batch_size):
    """小批量随机梯度下降"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

In [14]:
lr = 0.033  # 学习率
num_epochs = 10  # 训练轮数
batch_size = 100  # 批量大小

# 实例化模型和损失函数
net = linreg
loss = squared_loss

# 训练循环
for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, lables):
        # 前向传播：计算小批量损失
        l = loss(net(X, w, b), y)  # l的形状是(batch_size, 1)
        
        # 反向传播：计算梯度
        l.sum().backward()  # 求和后反向传播
        
        # 更新参数：使用SGD
        sgd([w, b], lr, batch_size)
    
    # 每个epoch结束后评估整个数据集上的损失
    with torch.no_grad():
        train_loss = loss(net(features, w, b), lables)
        print(f'epoch {epoch + 1}, loss {float(train_loss.mean()):f}')

print(f'\n真实w: {true_w}')
print(f'学习到的w: {w.reshape(-1).detach().numpy()}')
print(f'真实b: {true_b}')
print(f'学习到的b: {b.item():.4f}')

epoch 1, loss 8.437332
epoch 2, loss 4.339402
epoch 3, loss 2.232896
epoch 4, loss 1.149511
epoch 5, loss 0.592131
epoch 6, loss 0.305232
epoch 7, loss 0.157448
epoch 8, loss 0.081277
epoch 9, loss 0.041986
epoch 10, loss 0.021718

真实w: tensor([ 2.0000, -3.4000])
学习到的w: [ 1.9051145 -3.276562 ]
真实b: 4.2
学习到的b: 4.0598
